# Gravitational Wave Data Analysis — Hands-On Session

## Exploring GW250114_082203 with Open Data

Welcome! In this hands-on session we will work with **real gravitational-wave data** from the LIGO, Virgo, and KAGRA detectors. Our target is the event **GW250114_082203**, a compact binary coalescence detected during the O4 observing run.

### What we'll do on this tutorial

1. **Discover open data** by querying the [GWOSC](https://gwosc.org) catalogs and finding the loudest events
2. **Download open data** for both LIGO detectors (Hanford H1 and Livingston L1)
3. **Inspect the raw strain data** in the time domain
4. **Estimate the noise Power Spectral Density (PSD)** using Welch's method
5. **Whiten and band-pass** the data to reveal the gravitational-wave signal

By the end of this notebook, you should be able to **see the GW signal with your own eyes** in the whitened data from both detectors!

### 🧪 Interactive exercises

Throughout the tutorial you will find **3 hands-on exercises**. Each one has a hint and a collapsed solution block — try to work through it before expanding the solution!

| Exercise | Topic | After section |
|---|---|---|
| **Exercise 1** | Welch's trade-off: frequency resolution vs. variance | §5 PSD |
| **Exercise 2** | Implement whitening yourself from scratch | §6 Whitening |
| **Exercise 3** | Measure the H1–L1 arrival time difference | §7 Signal reveal |

> **Note**: This notebook is adapted from the [GW Open Data Workshop](https://github.com/gw-odw/odw) tutorials. All data used are publicly available through GWOSC.

## 1. Setup: Importing the tools we need

We will use three main Python packages:

- **[GWpy](https://gwpy.readthedocs.io/)** — the go-to package for GW data access and manipulation, built on an object-oriented design
- **[PyCBC](https://pycbc.org/)** — a powerful toolkit used in real GW searches and parameter estimation
- **[GWOSC](https://pypi.org/project/gwosc/)** — a lightweight client to query the GW Open Science Center catalog

We'll also use **NumPy**, **SciPy**, and **Matplotlib** for numerics and plotting.</VSCode.Cell>


In [ ]:
# Suppress harmless warnings
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

# Core scientific packages
import numpy as np
import scipy
import matplotlib.pyplot as plt

# GW-specific packages
import gwosc
print(f"GWOSC version: {gwosc.__version__}")
import gwpy
print(f"GWpy version: {gwpy.__version__}")
import pycbc
print(f"PyCBC version: {pycbc.__version__}")

# For nice plots
#%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 2. Discovering Open Data with GWOSC

Before downloading anything, let's explore **what data is actually available**. The [GWOSC](https://gwosc.org) provides a rich catalog of gravitational-wave events, and the `gwosc` Python package lets us query it programmatically.

The `gwosc.datasets` module lets us search for:
- **Catalogs** — collections of confidently detected events (e.g. GWTC-1, GWTC-2, GWTC-3, ...)
- **Events** — individual detections, with their GPS times and physical parameters
- **Runs** — the full strain-data releases from each observing run (O1, O2, O3, O4, ...)

Let's start by listing all the catalogs currently available.

In [ ]:
from gwosc.datasets import find_datasets

# List all available event catalogs
catalogs = find_datasets(type="catalog")
print("Available catalogs on GWOSC:")
for cat in catalogs:
    print(f"  • {cat}")


### Observing runs and event counts

Each catalog corresponds roughly to one or more observing runs. We can also list the **full strain-data releases** for each run, and count how many events live in a given catalog.

The most recent catalogs (GWTC-4, from the first half of O4) contain the largest number of events, reflecting the improved detector sensitivity in O4.

In [ ]:
from gwosc import datasets

# List the large strain-data releases from the observing runs
runs = find_datasets(type="run")
print("Full strain-data releases (observing runs):")
print(f"  {runs}\n")

# Count the events in a few representative catalogs
for catalog in ["GWTC-1-confident", "GWTC-2.1-confident", "GWTC-3-confident"]:
    events = datasets.find_datasets(type="events", catalog=catalog)
    print(f"{catalog}: {len(events)} events")


### Finding the loudest events

For this tutorial we want an event where the gravitational-wave signal is **as clear as possible** — that is, one with a high **signal-to-noise ratio (SNR)**. The louder the event, the easier it will be to see the chirp by eye once we whiten the data.

The `query_events` function lets us filter events by their physical/detection parameters, just like the [GWOSC Event Portal](https://gwosc.org/eventapi/html/query/). Let's search for the events detected with a **network matched-filter SNR ≥ 30** — the loudest signals ever observed.

In [ ]:
from gwosc.datasets import query_events

# Find all events with a network matched-filter SNR >= 30
loud_events = query_events(select=["network-matched-filter-snr >= 30"])

print("Events with network matched-filter SNR >= 30:")
for ev in sorted(set(loud_events)):
    print(f"  • {ev}")


In [ ]:
# To find the *single loudest* event, we progressively raise the SNR threshold
# and see which event survives to the highest value.
for threshold in [30, 40, 50, 60, 70, 80]:
    evs = sorted(set(query_events(select=[f"network-matched-filter-snr >= {threshold}"])))
    # Strip version suffixes (e.g. "-v1") for a cleaner display
    names = sorted({ev.rsplit("-v", 1)[0] for ev in evs})
    print(f"SNR >= {threshold:>2}:  {names}")


### Our target: GW250114_082203

As we raise the SNR threshold, the list shrinks until a single event survives above the rest: **GW250114_082203**, an O4b binary-black-hole merger and one of the **loudest gravitational-wave signals ever observed** (network SNR ≳ 70).

Because it is so loud, its chirp will be exceptionally clear once we whiten the data — which makes it the perfect event for this hands-on session. Let's download its strain data next.

## 3. Downloading Open Data from GWOSC

### The event: GW250114_082203

The event name follows the standard LIGO-Virgo-KAGRA convention: **GW** _YYMMDD_HHMMSS_. So **GW250114_082203** was detected on **January 14, 2025 at 08:22:03 UTC**, during the second half of the O4 observing run (O4b).

We'll download data from both LIGO detectors:
- **H1** — LIGO Hanford (Washington, USA)
- **L1** — LIGO Livingston (Louisiana, USA)

We'll fetch 32 seconds of data centred on the event time (16 seconds before and after). The `TimeSeries.fetch_open_data` method from GWpy handles the download automatically from the GWOSC server.

In [ ]:
from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries

# Get the GPS time of the event
gps = event_gps('GW250114_082203')
print(f"GPS time of GW250114_082203: {gps}")

# Define a 32-second window around the event (±16 s)
segment = (int(gps) - 16, int(gps) + 16)
print(f"Data segment: {segment} (GPS seconds)")

# The event may have a fractional GPS offset, so compute its exact position
# within our segment (in seconds from segment start)
event_time_offset = gps - int(gps)  # fractional part of GPS time
event_time_in_segment = 16.0 + event_time_offset
print(f"Event at {event_time_in_segment:.3f} s within the {segment[1]-segment[0]} s segment")

In [ ]:
# Download data for both LIGO detectors
# This may take a minute or two, depending on your internet connection
print("\nDownloading H1 (Hanford) data...")
hdata = TimeSeries.fetch_open_data('H1', *segment, verbose=True)

print("\nDownloading L1 (Livingston) data...")
ldata = TimeSeries.fetch_open_data('L1', *segment, verbose=True)

print("\n✓ Data successfully downloaded!")
print(f"  H1: {hdata}")
print(f"  L1: {ldata}")

print("\nStoring data to local files...")
hdata.write("GW250114_082203_H1.txt")
ldata.write("GW250114_082203_L1.txt")
print("✓ Data successfully stored!")

In [ ]:
## You can also load the data from local files if you have already downloaded them:
hdata = TimeSeries.read("GW250114_082203_H1.txt")
ldata = TimeSeries.read("GW250114_082203_L1.txt")

## 4. Inspecting the Raw Data

Let's look at the raw strain data in the time domain. The strain $h(t)$ is a dimensionless quantity measuring the fractional change in distance between the interferometer's test masses:

$$h(t) = \frac{\Delta L(t)}{L}$$

Typical GW signals produce strains of order $h \sim 10^{-21}$ to $10^{-22}$, while the detector noise is typically $10^{-22}$ to $10^{-23}$ per $\sqrt{\text{Hz}}$. This means that **raw strain data is completely dominated by noise** — you cannot see the signal by eye!

In [ ]:
# Let's check the basic properties of our data
T_h = hdata.duration.value   # total duration in seconds
dt_h = hdata.dt.value         # time step (1 / sampling frequency)
fs_h = 1.0 / dt_h             # sampling frequency in Hz

T_l = ldata.duration.value
dt_l = ldata.dt.value
fs_l = 1.0 / dt_l

print(f"H1 — Duration: {T_h} s,  dt: {dt_h:.6f} s,  fs: {fs_h:.0f} Hz")
print(f"L1 — Duration: {T_l} s,  dt: {dt_l:.6f} s,  fs: {fs_l:.0f} Hz")


In [ ]:
# Plot the raw strain data for both detectors

# H1 (Hanford)
plot_h = hdata.plot( color='gwpy:ligo-hanford')
plot_h.axes[0].set_title('LIGO Hanford (H1) — Raw Strain')
plot_h.axes[0].set_ylabel('Strain')

# L1 (Livingston)
plot_l = ldata.plot( color='gwpy:ligo-livingston')
plot_l.axes[0].set_title('LIGO Livingston (L1) — Raw Strain')
plot_l.axes[0].set_xlabel(f'GPS Time (seconds) — Event at {gps}')
plot_l.axes[0].set_ylabel('Strain')

plt.tight_layout()


As expected, the raw data looks like noise — the GW signal is completely buried. The data oscillates wildly because the dominant noise power lives at low frequencies (seismic noise below ~30 Hz) and high frequencies (shot noise above a few kHz). To reveal the signal, we need to:

1. **Characterise the noise** → estimate the Power Spectral Density (PSD)
2. **Whiten the data** → rescale by the PSD to flatten the noise spectrum
3. **Band-pass the data** → keep only the frequency range where the signal lives


## 5. The Power Spectral Density (PSD)

### What is the PSD?

The noise in a GW detector is a **stochastic process** $n(t)$. For stationary noise (noise whose statistical properties don't change over time), the **Power Spectral Density** $S_n(f)$ captures how the noise power is distributed across frequencies:

$$S_{n}(f) = \lim_{T \rightarrow +\infty} \frac{2}{T} \left\langle \left| \tilde{n}_{T}(f) \right|^{2} \right\rangle$$

where $\tilde{n}_T(f)$ is the Fourier transform of a noise segment of length $T$, and $\langle \cdot \rangle$ denotes an ensemble average.

A crucial property of stationary Gaussian noise: **different frequency bins are statistically independent**, with variance:

$$\langle \tilde{n}(f) \tilde{n}^*(f') \rangle = \frac{1}{2} S_n(f) \, \delta(f - f')$$

The square root of the PSD is called the **Amplitude Spectral Density (ASD)** and has units of $\text{strain} / \sqrt{\text{Hz}}$. The ASD is the standard way to visualise detector sensitivity.

### Estimating the PSD with Welch's method

Since we only have one realisation of the noise (not an ensemble), we estimate the PSD using **[Welch's method](https://en.wikipedia.org/wiki/Welch%27s_method)**:

1. Split the data into overlapping segments
2. Apply a window function to each segment (to reduce spectral leakage)
3. Compute the FFT of each segment
4. Average the squared magnitudes of all segments

There is a **trade-off**: longer segments give better frequency resolution ($\Delta f = 1/T_{\text{seg}}$), but fewer segments means more statistical fluctuations in the estimate.

In [ ]:
# Compute PSD using Welch's method with scipy
# nperseg controls segment length: 16384 samples → frequency resolution ≈ fs/8192
fs = fs_h  # Use H1 sampling rate (should be same for both detectors)

freqs_h, psd_h = scipy.signal.welch(hdata.value, fs, window='hann', nperseg=8192)
freqs_l, psd_l = scipy.signal.welch(ldata.value, fs, window='hann', nperseg=8192)

print(f"PSD frequency bins: {len(freqs_h)}")
print(f"Frequency resolution: {freqs_h[1] - freqs_h[0]:.3f} Hz")


In [ ]:
# Plot the Amplitude Spectral Density (ASD = sqrt(PSD)) for both detectors
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Skip the zero-frequency bin (freqs[0] = 0)
ax.loglog(freqs_h[1:], np.sqrt(psd_h[1:]), label='LIGO Hanford (H1)', color='gwpy:ligo-hanford')
ax.loglog(freqs_l[1:], np.sqrt(psd_l[1:]), label='LIGO Livingston (L1)', color='gwpy:ligo-livingston')

ax.set_xlim(5, 2000)
ax.set_ylim(1e-24, 1e-17)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel(r'ASD (Strain / $\sqrt{\mathrm{Hz}}$)')
ax.set_title('Amplitude Spectral Density — LIGO O4 Sensitivity')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate interesting features
ax.axvspan(5, 30, alpha=0.1, color='red')
ax.annotate('Seismic\nnoise wall', xy=(15, 3e-24), fontsize=10, color='darkred')

# Highlight spectral lines
for fline in [60, 120, 180, 500]:
    ax.axvline(fline, color='gray', alpha=0.3, linestyle='--', linewidth=0.5)

plt.tight_layout()


### Interpreting the ASD

The ASD plot reveals the characteristic sensitivity curve of the LIGO detectors in O4:

- **Below ~20–30 Hz**: Seismic noise dominates — the ground shakes too much
- **~30–200 Hz**: The most sensitive band — where most GW signals are detected
- **~200–1000 Hz**: Thermal noise from the mirror suspensions and coatings
- **Above ~1000 Hz**: Shot noise from photon counting statistics dominates

The sharp vertical lines (e.g. at 60, 120, 180 Hz) are **spectral lines** — narrow-band noise from the electric power grid (60 Hz in the US) and its harmonics, plus calibration lines deliberately injected to track the detector response.

> **Note**: The two LIGO detectors (H1 and L1) have nearly identical design, but their noise curves differ slightly due to different local environments (Hanford, WA vs. Livingston, LA).


---
## 🧪 Exercise 1: Exploring Welch's trade-off

You just computed the ASD with `nperseg=4096`. Recall the key trade-off in Welch's method:

- **Shorter segments** → more overlapping windows → smoother, less noisy estimate — but coarser frequency resolution ($\Delta f = f_s / N_\text{seg}$)
- **Longer segments** → finer frequency resolution — but fewer averages → noisier estimate

**Your task:** compute the ASD for H1 with `nperseg=256` and `nperseg=65536` and plot all three curves on the same log-log axes. Observe:
- Which setting resolves the 60 Hz power-line spike best?
- Which is smoothest overall?

> **Hint:** Just call `scipy.signal.welch(hdata.value, fs, window='hann', nperseg=...)` twice. Reuse the existing `freqs_h, psd_h` arrays for the middle curve (no need to recompute). `fs` and `hdata` are already available.

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
freqs_256, psd_256 = scipy.signal.welch(hdata.value, fs, window='hann', nperseg=256)
freqs_64k, psd_64k = scipy.signal.welch(hdata.value, fs, window='hann', nperseg=65536)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(freqs_256[1:], np.sqrt(psd_256[1:]), alpha=0.7,
          label=f'nperseg=256   (Δf = {fs/256:.1f} Hz, many averages → smooth but coarse)')
ax.loglog(freqs_h[1:],   np.sqrt(psd_h[1:]),   alpha=0.9, color='C1',
          label=f'nperseg=4096  (Δf = {fs/4096:.2f} Hz, default)')
ax.loglog(freqs_64k[1:], np.sqrt(psd_64k[1:]), alpha=0.7,
          label=f'nperseg=65536 (Δf = {fs/65536:.4f} Hz, few averages → noisy but fine)')
ax.set_xlim(5, 2000)
ax.set_ylim(1e-24, 1e-17)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel(r'ASD (Strain / $\sqrt{\mathrm{Hz}}$)')
ax.set_title("Welch's trade-off: frequency resolution vs. variance — H1")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
# Notice: nperseg=65536 resolves the 60 Hz line as a sharp spike,
# while nperseg=256 smears it across a wide bin.
```

</details>

In [ ]:
# ── Exercise 1 — your workspace ───────────────────────────────────────────────
# Compute the ASD with two different nperseg values and compare with the default.

# nperseg = 256  (many short segments → coarser Δf, smoother curve)
# freqs_256, psd_256 = scipy.signal.welch(hdata.value, fs, window='hann', nperseg=256)

# nperseg = 32768  (few long segments → finer Δf, noisier curve)
# freqs_64k, psd_64k = scipy.signal.welch(hdata.value, fs, window='hann', nperseg=65536)

# Plot all three on the same log-log axes. Reuse freqs_h, psd_h for the middle curve.


## 6. Whitening and Band-passing the Data

### Why whiten?

The raw strain data is **colored noise**: different frequencies have vastly different power (as we just saw in the ASD plot). This makes it impossible to see a GW signal by eye.

**Whitening** rescales the data in the frequency domain so that all frequency bins have the same variance:

$$\tilde{d}_j^{\rm w} \equiv \sqrt{\frac{2 \Delta f}{S_n(f_j)}} \, \tilde{d}_j$$

After whitening, the noise becomes **white** — every frequency bin contributes equally. The Fourier bins of white noise are independent Gaussian variables with $\sigma = 1$. In the code below we implement the same idea using NumPy's discrete `rfft/irfft` convention, where the corresponding whitening factor is `ASD / \sqrt{2\,dt}`.

### Why band-pass?

Even after whitening, data at very low frequencies (dominated by seismic noise) and very high frequencies (dominated by shot noise, and where the signal has negligible power) adds no useful information about the GW signal. We apply a **band-pass filter** to keep only the frequency range where the signal lives:

$$\tilde{d}_j^{\rm wbp}(f) \equiv \tilde{w}_{\rm bp}(f) \, \tilde{d}_j^{\rm w}(f)$$

where $\tilde{w}_{\rm bp}(f)$ smoothly tapers to zero at the band edges.

---
## 🧪 Exercise 2: Implement whitening from scratch

The whitening formula is conceptually simple: **divide the Fourier transform of the data by the ASD**. The helper function `whiten_bandpass` below does this plus several extras (edge tapering, band-pass, etc.). Let's first build the core whitening step ourselves.

You already have, from Section 5:
- `hdata` — the raw H1 `TimeSeries`
- `freqs_h`, `psd_h` — the Welch PSD estimate (numpy arrays)

**Your task:** whiten `hdata` in ~6 lines of NumPy, following these steps:

1. Apply `scipy.signal.windows.tukey(N, alpha=0.1)` to the data before the FFT (avoids edge ringing)
2. `np.fft.rfft(data_windowed)` → complex Fourier coefficients
3. `np.fft.rfftfreq(N, d=dt)` → the matching positive-frequency grid
4. `np.interp(freqs_fd, freqs_h, np.sqrt(psd_h))` → ASD interpolated onto that grid
5. Divide the FFT by the whitening norm and `np.fft.irfft(..., n=N)` back to the time domain
6. Plot raw vs whitened: the whitened noise should look flat and have RMS ≈ 1

> **⚠️ Normalization gotcha:** NumPy's `rfft` carries **no `dt` factor**, but the ASD is in physical units (strain/√Hz). The correct whitening divisor is `asd / np.sqrt(2 * dt)`, **not** bare `asd` — using the bare ASD inflates the result by a factor of `1/sqrt(2·dt) ≈ 45` and gives RMS >> 1.

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
N_ex  = len(hdata)
dt_ex = hdata.dt.value

# Tukey window to suppress edge transients
tukey_win     = scipy.signal.windows.tukey(N_ex, alpha=0.1)
data_windowed = tukey_win * hdata.value

# Step 1: FFT
data_fd_ex   = np.fft.rfft(data_windowed)

# Step 2: Frequency grid
freqs_fd_ex  = np.fft.rfftfreq(N_ex, d=dt_ex)

# Step 3: Interpolate ASD and build whitening norm.
# numpy rfft has no dt factor → divide by ASD/sqrt(2*dt), not just ASD.
asd_ex       = np.interp(freqs_fd_ex, freqs_h, np.sqrt(psd_h),
                          left=np.inf, right=np.inf)
whiten_norm  = asd_ex / np.sqrt(2 * dt_ex)
whiten_norm[0] = np.inf   # DC → 0 after division

# Step 4: Whiten
data_whitened_fd = data_fd_ex / whiten_norm
data_whitened    = np.fft.irfft(data_whitened_fd, n=N_ex)
times_ex         = dt_ex * np.arange(N_ex)

# Step 5: Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(times_ex, hdata.value, color='gray', linewidth=0.4)
axes[0].set_ylabel('Strain')
axes[0].set_title('Raw H1 strain — dominated by low-frequency seismic noise')
axes[1].plot(times_ex, data_whitened, color='gwpy:ligo-hanford', linewidth=0.4)
axes[1].set_ylabel('Whitened strain')
axes[1].set_title('Whitened H1 strain (no band-pass) — noise is now spectrally flat')
axes[1].set_xlabel('Time (s from start of segment)')
plt.tight_layout()

# Step 6: Plot around the event
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(times_ex, hdata.value, color='gray', linewidth=0.4)
axes[0].set_ylabel('Strain')
axes[0].set_title('Raw H1 strain — dominated by low-frequency seismic noise')
axes[1].plot(times_ex, data_whitened, color='gwpy:ligo-hanford', linewidth=0.4)
axes[1].set_ylabel('Whitened strain')
axes[1].set_title('Whitened H1 strain (no band-pass) — noise is now spectrally flat')
axes[1].set_xlabel('Time (s from start of segment)')
axes[1].set_xlim(16.0,16.25)
plt.tight_layout()

print(f"Whitened RMS ≈ {np.std(data_whitened):.2f}  (expected ~1 for unit-normalised whitening)")
# The next cell (whiten_bandpass) adds a band-pass on top of this.
```

</details>

In [ ]:
# ── Exercise 2 — your workspace ───────────────────────────────────────────────
# Implement basic whitening for H1 using the PSD estimated in Section 5.
# (No band-pass yet — just the core whitening step.)

N_ex  = len(hdata)
dt_ex = hdata.dt.value

# Step 0: Tukey window to suppress edge ringing before the FFT
# tukey_win     = scipy.signal.windows.tukey(N_ex, alpha=0.1)
# data_windowed = tukey_win * hdata.value

# Step 1: FFT of windowed data
# data_fd_ex = # Here we do the real FFT of the windowed data (use np.fft.rfft)

# Step 2: Frequency array matching this FFT
# freqs_fd_ex = # Here we compute the frequency bins for the real FFT (use np.fft.rfftfreq)

# Step 3: Interpolate ASD and build whitening norm.
# ⚠️  numpy rfft has no dt factor → correct norm is ASD / sqrt(2 * dt), not bare ASD!
# asd_ex      = np.interp(freqs_fd_ex, freqs_h, np.sqrt(psd_h), left=np.inf, right=np.inf)
# whiten_norm = # here we compute the whitening norm 
# whiten_norm[0] = np.inf  # suppress DC

# Step 4: Whiten and transform back
# data_whitened_fd = data_fd_ex / whiten_norm
# data_whitened    = # Here we do the inverse real FFT to get back to time domain (use np.fft.irfft)
# times_ex         = dt_ex * np.arange(N_ex)

# Step 5: Plot raw vs whitened — the RMS should be close to 1
# print(f"Whitened RMS ≈ {np.std(data_whitened):.2f}  (should be close to 1)")
# Here we plot the raw and whitened data on the same axes for comparison (use plt.subplots and ax.plot). 
# After checking for the whole duration, we use a time window around the event (e.g., 16–16.25 s) to zoom in on the signal.

In [ ]:
# Helper function: smooth band-pass window (Tukey-like tapering)
def bandpass_window(freqs, f_low, f_high, taper_width_low=5., taper_width_high=50.):
    """
    Create a band-pass window that smoothly tapers to zero at the edges.
    
    Parameters:
    - freqs: array of frequencies
    - f_low, f_high: passband limits (Hz)
    - taper_width_low, taper_width_high: width of the cosine taper at each edge (Hz)
    
    Returns:
    - window: array of same shape as freqs, values in [0, 1]
    """
    window = np.ones_like(freqs)
    
    # Taper at low frequencies: 0 at f_low, 1 at f_low + taper_width_low
    mask_low = (freqs > f_low) & (freqs < f_low + taper_width_low)
    window[mask_low] = 0.5 * (1 - np.cos(np.pi * (freqs[mask_low] - f_low) / taper_width_low))
    
    # Set to zero below f_low
    window[freqs <= f_low] = 0.0
    
    # Taper at high frequencies: 1 at f_high - taper_width_high, 0 at f_high
    mask_high = (freqs > f_high - taper_width_high) & (freqs < f_high)
    window[mask_high] = 0.5 * (1 + np.cos(np.pi * (freqs[mask_high] - (f_high - taper_width_high)) / taper_width_high))
    
    # Set to zero above f_high
    window[freqs >= f_high] = 0.0
    
    return window


def whiten_bandpass(data_timeseries, asd_func, f_low=20, f_high=400, 
                    taper_low=10., taper_high=100.):
    """
    Whiten and band-pass a GWpy TimeSeries.
    
    Parameters:
    - data_timeseries: GWpy TimeSeries object
    - asd_func: interpolation function for the ASD (Hz → ASD value)
    - f_low, f_high: band-pass limits (Hz)
    - taper_low, taper_high: taper widths (Hz)
    
    Returns:
    - times: time array (seconds from start)
    - whitened_td: whitened, band-passed time-domain data
    - window_bp: the band-pass window used
    - freqs_fd: the positive-frequency grid used in the FFT
    """
    N = len(data_timeseries)
    dt = data_timeseries.dt.value
    
    # Apply a time-domain Tukey window to limit edge transients before the FFT
    tukey_win = scipy.signal.windows.tukey(N, alpha=0.1)
    data_windowed = tukey_win * data_timeseries.value
    
    # Use the standard NumPy FFT convention for time-domain whitening.
    # With this convention, dividing by ASD / sqrt(2*dt) gives a whitened
    # time series whose unfiltered samples have RMS close to 1 for stationary noise.
    data_fd = np.fft.rfft(data_windowed)
    freqs_fd = np.fft.rfftfreq(N, d=dt)
    whiten_norm = asd_func(freqs_fd) / np.sqrt(2 * dt)
    
    # Build the band-pass window and suppress frequencies outside the signal band.
    window_bp = bandpass_window(freqs_fd, f_low, f_high, taper_low, taper_high)
    data_wbp = window_bp * data_fd / whiten_norm
    
    # Transform back to the time domain using the matching inverse FFT convention.
    data_wbp_td = np.fft.irfft(data_wbp, n=N)
    times = dt * np.arange(len(data_wbp_td))
    
    return times, data_wbp_td, window_bp, freqs_fd

In [ ]:
# Compute ASDs using GWpy's built-in method (for the interpolation functions)
asd_h = hdata.asd(fftlength=2., method="median")
asd_l = ldata.asd(fftlength=2., method="median")

# Create interpolation functions for the ASDs
from scipy.interpolate import interp1d
asd_h_interp = interp1d(asd_h.frequencies.value, asd_h.value, 
                         bounds_error=False, fill_value=np.inf)
asd_l_interp = interp1d(asd_l.frequencies.value, asd_l.value,
                         bounds_error=False, fill_value=np.inf)

# Whiten and band-pass both detectors
# For a compact binary merger, the interesting frequencies are ~20–500 Hz
print("Whitening H1 data...")
times_h, h_wbp, window_h, freqs_fd_h = whiten_bandpass(
    hdata, asd_h_interp, f_low=20, f_high=500, taper_low=10., taper_high=100.
)

print("Whitening L1 data...")
times_l, l_wbp, window_l, freqs_fd_l = whiten_bandpass(
    ldata, asd_l_interp, f_low=20, f_high=500, taper_low=10., taper_high=100.
)

# For white noise, band-passing reduces the time-domain RMS by roughly sqrt(mean(window^2)).
expected_rms_h = np.sqrt(np.mean(window_h**2))
expected_rms_l = np.sqrt(np.mean(window_l**2))

print("✓ Whitening complete!")
print(f"  H1 whitened data shape: {h_wbp.shape}")
print(f"  L1 whitened data shape: {l_wbp.shape}")
print(f"  H1 filtered RMS: {np.std(h_wbp):.3f}  (expected ≈ {expected_rms_h:.3f})")
print(f"  L1 filtered RMS: {np.std(l_wbp):.3f}  (expected ≈ {expected_rms_l:.3f})")
print("  RMS < 1 is expected here: whitening makes the full-band noise unit scale, and band-passing removes part of that variance.")


### Sanity check: the ASD and FFT normalization

Before looking at the time domain, let's verify that our ASD estimate is consistent with the Fourier-domain data. For the convention `dt × rfft(data)`, stationary noise should satisfy

$$|\tilde{d}(f)| \sim \frac{\mathrm{ASD}(f)}{\sqrt{2\,\Delta f}}$$

up to random scatter from one frequency bin to the next. This is a **frequency-domain PSD check**. The time-domain whitening used below adopts a different normalization, chosen so that the unfiltered whitened samples have RMS close to 1 for stationary noise; after the 20–500 Hz band-pass, the RMS becomes smaller because only part of the full band is retained.

### How to interpret the filtered RMS

At this stage there are **two different but compatible sanity checks**:

1. In the **frequency domain**, the PSD-normalized spectrum should sit at order unity across the passband.
2. In the **time domain**, the *unfiltered* whitened noise would have RMS close to 1.
3. After the **20–500 Hz band-pass**, the RMS drops below 1 because we deliberately discard part of the noise power along with irrelevant frequencies.

For this notebook, the filtered RMS should be roughly set by $\sqrt{\langle w_{\rm bp}^2 \rangle}$, where $w_{\rm bp}(f)$ is the band-pass window. That gives a useful rule of thumb for deciding whether the whitening and filtering behave as expected.

In [ ]:
# Reconstruct the frequency-domain amplitudes for a PSD normalization check
N_h = len(hdata)
dt_h_val = hdata.dt.value
df_h = 1.0 / (N_h * dt_h_val)

# Use dt * rfft here because the one-sided PSD convention is
# S_n(f) ≈ (2 / T) |dt * rfft(data)|^2
tukey_h = scipy.signal.windows.tukey(N_h, alpha=0.1)
h_fd = np.fft.rfft(tukey_h * hdata.value) * dt_h_val
freqs_fd_h = np.fft.rfftfreq(N_h, d=dt_h_val)
h_fd_psdnorm = window_h * h_fd / (asd_h_interp(freqs_fd_h) / np.sqrt(2 * df_h))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before PSD normalization
axes[0].loglog(freqs_fd_h[1:], np.abs(h_fd[1:]), alpha=0.5, color='gray')
axes[0].loglog(asd_h.frequencies.value[1:], asd_h.value[1:] / np.sqrt(2 * df_h), 
               'C0', label='ASD / √(2Δf)')
axes[0].set_xlim(10, 2000)
axes[0].set_ylim(1e-24, 1e-20)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('|dt × FFT| (strain s)')
axes[0].set_title('Before PSD Normalization — H1')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# After PSD normalization and band-limiting
axes[1].loglog(freqs_fd_h[1:], np.abs(h_fd_psdnorm[1:]), alpha=0.5, color='gray')
axes[1].axhline(1.0, color='C1', linestyle='--', label='Order-unity noise level')
axes[1].set_xlim(10, 2000)
axes[1].set_ylim(1e-2, 10)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel(r'|dt × FFT| / [ASD / $\sqrt{2\Delta f}$]')
axes[1].set_title('PSD-Normalized Spectrum & Band-pass — H1')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

## 7. Revealing the Gravitational-Wave Signal

Now for the exciting part! The whitened, band-passed time-domain data should show the GW signal clearly above the noise.

Remember: in whitened data, the noise is **white** before band-limiting. After we additionally restrict the data to 20–500 Hz, the residual time-domain noise has an RMS below 1 because we have removed most of the available frequency band. Any coherent structure that emerges on top of that filtered noise is a real feature of the data. A GW signal will appear as a **coherent oscillation that grows in amplitude and frequency**, peaking at the merger time, then ringing down.

In [ ]:
# Plot the whitened, band-passed time-domain data for BOTH detectors
# The event GPS time is at event_time_in_segment seconds from the start

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# H1
axes[0].plot(times_h, h_wbp, color='gwpy:ligo-hanford', linewidth=0.6)
axes[0].axvline(x=event_time_in_segment, color='red', linestyle='--', alpha=0.7, label=f'Event time (GPS {gps})')
axes[0].set_ylabel('Filtered whitened strain')
axes[0].set_title('LIGO Hanford (H1) — Whitened & Band-passed [20–500 Hz]')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# L1
axes[1].plot(times_l, l_wbp, color='gwpy:ligo-livingston', linewidth=0.6)
axes[1].axvline(x=event_time_in_segment, color='red', linestyle='--', alpha=0.7, label=f'Event time (GPS {gps})')
axes[1].set_xlabel('Time (seconds from start of segment)')
axes[1].set_ylabel('Filtered whitened strain')
axes[1].set_title('LIGO Livingston (L1) — Whitened & Band-passed [20–500 Hz]')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Gravitational-Wave Signal in Whitened Data — GW250114_082203', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()


In [ ]:
# Zoom in around the event time (±1 second)
zoom_halfwidth = 0.25
zoom_start = event_time_in_segment - zoom_halfwidth
zoom_end = event_time_in_segment + zoom_halfwidth

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(times_h, h_wbp, color='gwpy:ligo-hanford', linewidth=0.8)
axes[0].axvline(x=event_time_in_segment, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlim(zoom_start, zoom_end)
axes[0].set_ylabel('Filtered whitened strain')
axes[0].set_title('H1 — Zoom around the merger')
axes[0].grid(True, alpha=0.3)

axes[1].plot(times_l, l_wbp, color='gwpy:ligo-livingston', linewidth=0.8)
axes[1].axvline(x=event_time_in_segment, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlim(zoom_start, zoom_end)
axes[1].set_xlabel('Time (seconds from start of segment)')
axes[1].set_ylabel('Filtered whitened strain')
axes[1].set_title('L1 — Zoom around the merger')
axes[1].grid(True, alpha=0.3)

plt.suptitle('GW250114_082203 — Zoom on the Signal', fontsize=14, fontweight='bold')
plt.tight_layout()


In [ ]:
# Overlay both detectors for a direct comparison
fig, ax = plt.subplots(1, 1, figsize=(14, 5))

ax.plot(times_h, h_wbp, color='gwpy:ligo-hanford', linewidth=0.7, alpha=0.8, label='H1 (Hanford)')
ax.plot(times_l, -l_wbp, color='gwpy:ligo-livingston', linewidth=0.7, alpha=0.8, label='L1 (Livingston, flipped)')
ax.axvline(x=event_time_in_segment, color='red', linestyle='--', alpha=0.5, label=f'Event time')

ax.set_xlim(event_time_in_segment - 0.15, event_time_in_segment + 0.1)
ax.set_xlabel('Time (seconds from start of segment)')
ax.set_ylabel('Whitened Strain')
ax.set_title('GW250114_082203 — H1 vs. L1 (L1 sign-flipped)', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()


---
## 🧪 Exercise 3: Measuring the H1–L1 arrival time difference

Gravitational waves travel at the speed of light. The LIGO detectors at Hanford and Livingston are separated by ~3000 km, so the maximum possible time difference between a signal arriving at one site vs. the other is:

$$\Delta t_{\max} = \frac{d}{c} = \frac{3000\,\text{km}}{3 \times 10^5\,\text{km/s}} \approx 10\,\text{ms}$$

**Your task:** estimate the actual H1–L1 delay for GW250114_082203. Two options:

- **Option A (by eye):** in the overlay plot above, narrow the x-limits further (e.g. `±0.05 s`) and compare where the H1 and L1 peaks fall.

- **Option B (cross-correlation):** extract a short window of both whitened series around the merger and use `scipy.signal.correlate` to find the lag that maximises the correlation. Convert the lag index to ms.

> **Hint for Option B:** `h_wbp`, `l_wbp`, `times_h`, `times_l`, `event_time_in_segment`, and `hdata.dt.value` are all already in memory. The cross-correlation lag array in samples is `np.arange(-(len(l_win)-1), len(h_win))`.

<details>
<summary><b>💡 Solution — click to expand</b></summary>

```python
from scipy.signal import correlate

# Short window around the merger
win_half = 0.10  # seconds
mask_h = (times_h >= event_time_in_segment - win_half) & (times_h <= event_time_in_segment + win_half)
mask_l = (times_l >= event_time_in_segment - win_half) & (times_l <= event_time_in_segment + win_half)
h_win, l_win = h_wbp[mask_h], l_wbp[mask_l]

# Cross-correlate
corr      = correlate(h_win, l_win, mode='full')
lag_samp  = np.arange(-(len(l_win) - 1), len(h_win))
lag_ms    = lag_samp * hdata.dt.value * 1000   # convert to milliseconds

best_lag_ms = lag_ms[np.argmax(np.abs(corr))]
print(f"Estimated H1–L1 time delay: {best_lag_ms:.2f} ms")
print(f"(Positive → H1 leads L1;  Negative → L1 leads H1)")
print(f"Physical upper limit for this baseline: ±10 ms")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lag_ms, corr / np.max(np.abs(corr)), linewidth=0.8)
ax.axvline(best_lag_ms, color='red', linestyle='--',
           label=f'Best lag: {best_lag_ms:.2f} ms')
ax.axvspan(-10, 10, alpha=0.10, color='green', label='Physical range (±10 ms)')
ax.set_xlabel('Lag (ms)')
ax.set_ylabel('Normalised cross-correlation')
ax.set_title('H1–L1 cross-correlation around merger — GW250114_082203')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
```

</details>

In [ ]:
# ── Exercise 3 — your workspace ───────────────────────────────────────────────
# Estimate the arrival time difference between H1 and L1.

# Option A (by eye): tighten the x-limits on the overlay plot above
#   and read off where the peaks of H1 and L1 are relative to each other.

# Option B (cross-correlation):
#   1. Extract a short window of both whitened series around the merger (±0.1 s)
#      mask_h = (times_h >= event_time_in_segment - 0.1) & (times_h <= event_time_in_segment + 0.1)
       # Do the same for Livingston
#      h_win = # Apply the mask to h_wbp
#      l_win = # Apply the mask to l_wbp
#   2. Cross-correlate with scipy.signal.correlate
#   3. Convert the lag index to milliseconds:  lag_ms = lag_samples * hdata.dt.value * 1000
#   4. Find the lag at maximum |correlation| → that's your H1–L1 delay estimate
#
# Expected: |delay| ≲ 10 ms  (Hanford–Livingston baseline is ~3000 km)


### What do we see?

The whitened, band-passed data reveals a clear **chirp-like signal** — the hallmark of a compact binary coalescence:

1. **Inspiral**: The signal starts at low amplitude and high frequency, slowly growing. The binary's orbit shrinks due to gravitational-wave emission.

2. **Merger**: A sharp peak in amplitude as the two objects plunge together.

3. **Ringdown**: A rapid decay as the merged remnant settles into a stable state.

We can see the signal in **both detectors** (H1 and L1). In the overlay plot, we flipped the sign of L1 — this is because the detectors have different orientations, and a GW signal can appear with opposite sign in different detectors (or even be absent in one, if the source is in a blind spot). The fact that we see a **coherent, consistent signal in both detectors** is a strong indicator that this is a real astrophysical event, not a detector glitch!

### Why does whitening work?

Without whitening, the data is coloured noise — low frequencies dominate the raw strain by orders of magnitude. The GW signal lives in a narrow band and is completely drowned out. After dividing by the ASD, every frequency contributes equally, and the signal — which is a coherent pattern across frequency — stands out above the now-uniform white noise background.</VSCode.Cell>


## 8. Spectrograms: a short and clear view of the chirp

For a quick pedagogical view, we will make spectrograms from the **already whitened and band-passed** data from Section 6. This suppresses low-frequency seismic noise and makes the chirp easier to isolate.

Minimal recipe:
1. Use short FFTs (`fftlength = 0.064 s`) and high overlap for better time tracking.
2. Zoom tightly around merger (`+-0.20 s`) and focus on the most informative band (`30-350 Hz`).
3. Plot **excess power**: divide each frequency bin by its median over time. This removes broad stationary background and highlights the transient chirp.
4. Use logarithmic color scaling (`LogNorm`) on this excess-power map.

This keeps the section short while making the signal track much easier to see in both detectors.

In [ ]:
# Compact, pedagogical spectrogram view focused on signal visibility
from matplotlib.colors import LogNorm
from gwpy.timeseries import TimeSeries

# Reuse filtered-whitened data from Section 6 to enhance transient visibility
h_wbp_ts = TimeSeries(h_wbp, dt=hdata.dt.value, t0=hdata.t0.value)
l_wbp_ts = TimeSeries(l_wbp, dt=ldata.dt.value, t0=ldata.t0.value)

fftlength = 0.064   # s -> better time localization for short BBH chirp
overlap = 0.058      # s -> smooth time evolution
fmin, fmax = 30, 350
twin = 0.20          # seconds around merger

def make_zoomed_spec(ts):
    spec = ts.spectrogram2(fftlength=fftlength, overlap=overlap, window='hann') ** 0.5
    times_rel = spec.times.value - gps
    tmask = (times_rel >= -twin) & (times_rel <= twin)
    z = spec.value[tmask, :]

    # Pedagogical trick: normalize each frequency bin by its time-median
    # to suppress stationary background and highlight transient excess power.
    floor = np.percentile(z[z > 0], 1) if np.any(z > 0) else 1e-12
    bg = np.median(z, axis=0, keepdims=True)
    z_excess = z / np.maximum(bg, floor)

    positive = z_excess[z_excess > 0]
    vmin = 0.8
    vmax = np.percentile(positive, 99.5) if positive.size else 5.0
    if vmax <= vmin:
        vmax = vmin * 5.0

    return spec, times_rel[tmask], z_excess, vmin, vmax

spec_h, t_h, z_h, vmin_h, vmax_h = make_zoomed_spec(h_wbp_ts)
spec_l, t_l, z_l, vmin_l, vmax_l = make_zoomed_spec(l_wbp_ts)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for ax, spec, tt, zz, vmin, vmax, title, cmap in [
    (axes[0], spec_h, t_h, z_h, vmin_h, vmax_h, 'H1 (Hanford)', 'magma'),
    (axes[1], spec_l, t_l, z_l, vmin_l, vmax_l, 'L1 (Livingston)', 'viridis'),
]:
    pcm = ax.pcolormesh(
        tt,
        spec.frequencies.value,
        zz.T,
        shading='auto',
        cmap=cmap,
        norm=LogNorm(vmin=vmin, vmax=vmax),
    )
    ax.axvline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.8, label='Merger')
    ax.set_yscale('log')
    ax.set_ylim(fmin, fmax)
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'{title} - spectrogram (excess over median)')
    ax.grid(True, alpha=0.15)
    ax.legend(loc='upper right')
    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label('excess sqrt(power) / median')

axes[1].set_xlabel('Time from merger (s)')
plt.suptitle('GW250114_082203 - chirp-enhanced spectrograms', fontweight='bold', y=1.01)
plt.tight_layout()

print('If the track is still faint: set twin=0.15 or lower fmax to 250 Hz.')

## 9. Summary

In this hands-on tutorial, we have:

| Step | What we did | Key takeaway |
|---|---|---|
| **1. Data discovery** | Queried GWOSC catalogs and filtered events by SNR | GW250114_082203 is the loudest event available — ideal for this tutorial |
| **2. Data access** | Downloaded 32 s of real LIGO strain for GW250114_082203 using GWpy | GWOSC provides free, open access to GW data |
| **3. Raw inspection** | Plotted the time-domain strain | Raw data is dominated by coloured noise; signals are invisible |
| **4. PSD estimation** | Computed the ASD using Welch's method | The PSD captures how noise power is distributed across frequencies |
| **🧪 Exercise 1** | Explored the Welch `nperseg` trade-off | Longer segments → finer frequency resolution but fewer averages → noisier estimate |
| **5. Whitening** | Divided the FFT by the ASD | Whitening flattens the noise spectrum so all frequencies have equal weight |
| **🧪 Exercise 2** | Implemented whitening from scratch | Whitening = rfft → divide by ASD → irfft; RMS of result ≈ 1 for stationary noise |
| **6. Band-passing** | Kept only 20–500 Hz with smooth tapers | Removing irrelevant frequencies reduces background fluctuations |
| **7. Signal reveal** | Plotted the whitened time-domain data | **The GW chirp signal is clearly visible in both H1 and L1!** |
| **🧪 Exercise 3** | Measured the H1–L1 arrival time difference | Cross-correlation gives |delay| ≲ 10 ms, consistent with the ~3000 km baseline |
| **8. Spectrogram** | Computed time-frequency spectrograms of both detectors | **The chirp track is visible as a sweeping curve in the time-frequency plane!** |

---

### Where to go next

- **`template_tutorial.ipynb`** ← the natural continuation of this notebook:
  - Generating IMR waveform templates (TD and FD, different approximants)
  - Matched filtering and the noise-weighted inner product
  - Intuitive visualisation of how SNR depends on time alignment
  - A short parameter-estimation run with bilby

- **`catalog_tutorial.ipynb`**: Explore the full GWTC-5.0 catalog — population plots, error bars, corner plots and prior-vs-posterior comparisons

- **Matched filtering deep-dive** (Tutorials 4.1–4.2 from the ODW): use PyCBC's template banks and significance estimation

---

*This notebook uses data from the Gravitational Wave Open Science Center (GWOSC), a service of LIGO Laboratory, the LIGO Scientific Collaboration, the Virgo Collaboration, and KAGRA.*